# CRAFT-GC: Stage B GPU Experiments (Google Colab)

**Setup (once):**
1. Runtime → **Change runtime type** → **T4 GPU**
2. Colab sidebar → **🔑 Secrets** → add secret named `HF_TOKEN` with your Hugging Face token → enable notebook access

**Run all cells.** Cell 1 will clone the GitHub repo automatically if needed.

**After finish:** download `results/` and `craft-gc-paper/figures/` to your machine.

In [ ]:
!pip install -q torch torchvision open-clip-torch diffusers transformers accelerate pandas matplotlib seaborn huggingface_hub

import os, sys, subprocess
from getpass import getpass

if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        os.environ["HF_TOKEN"] = getpass("HF Token (input hidden): ")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

REPO_URL = "https://github.com/Natiabay/Dynamic-Open-Set-Post-Processing-for-Intersectional-Fairness-in-Text-to-Image-Diffusion-Models.git"
PROJECT = "/content/Research"

if os.path.isdir(PROJECT):
    subprocess.run(["git", "-C", PROJECT, "pull"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, PROJECT], check=True)

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
# Install package so `import craft_gc` works in all scripts
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", PROJECT], check=True)

import torch
print("CWD:", os.getcwd())
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
print("craft_gc import:", end=" ")
import craft_gc
print("OK", craft_gc.__version__)

In [ ]:
!python scripts/evaluate_embeddings.py --device cuda
!python scripts/merge_paper_results.py
!python scripts/plot_figures.py

In [ ]:
# Stage B: SD image generation — expect 2–4 HOURS (not minutes) for 50 prompts × 4 methods × 5 seeds
import os, sys, subprocess
import torch

PROJECT = "/content/Research"
os.chdir(PROJECT)

if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Runtime → Change runtime type → T4 GPU → Save → re-run from Cell 1")

for pkg in ("diffusers", "transformers", "accelerate", "craft_gc"):
    __import__(pkg)
print("GPU:", torch.cuda.get_device_name(0))
print("Starting Stage B (Stable Diffusion download + 1000 images)...")

# Single script — stops on error (unlike chained ! commands)
subprocess.run(
    [sys.executable, "scripts/colab_run_stage_b.py", "--device", "cuda", "--limit", "50"],
    cwd=PROJECT,
    check=True,
)
print("Done. Download results/ and craft-gc-paper/figures/")